# Capítulo 6: Precognição (Pensando Passo a Passo)

- [Lição](#lesson)
- [Exercícios](#exercises)
- [Área de Testes](#example-playground)

## Configuração

Execute a célula de configuração abaixo para carregar sua chave de API e estabelecer a função auxiliar `get_completion`.

In [ ]:
!pip install anthropic

# Import python's built-in regular expression library
import re
import anthropic

# Retrieve the API_KEY & MODEL_NAME variables from the IPython store
%store -r API_KEY
%store -r MODEL_NAME

client = anthropic.Anthropic(api_key=API_KEY)

def get_completion(prompt: str, system_prompt="", prefill=""):
    message = client.messages.create(
        model=MODEL_NAME,
        max_tokens=2000,
        temperature=0.0,
        system=system_prompt,
        messages=[
          {"role": "user", "content": prompt},
          {"role": "assistant", "content": prefill}
        ]
    )
    return message.content[0].text

---

## Lição

Se alguém te acordasse e imediatamente começasse a te fazer várias perguntas complicadas que você tivesse que responder na hora, como você se sairia? Provavelmente não tão bem quanto se você tivesse tempo para **pensar na sua resposta primeiro**.

Adivinha? O Claude é da mesma forma.

**Dar ao Claude tempo para pensar passo a passo às vezes torna o Claude mais preciso**, particularmente para tarefas complexas. No entanto, **pensar só conta quando é em voz alta**. Você não pode pedir ao Claude para pensar mas produzir apenas a resposta - neste caso, nenhum pensamento realmente ocorreu.

### Exemplos

No prompt abaixo, está claro para um leitor humano que a segunda frase contradiz a primeira. Mas **o Claude leva a palavra "unrelated" (não relacionado) literalmente demais**.

In [ ]:
# Prompt
PROMPT = """Is this movie review sentiment positive or negative?

This movie blew my mind with its freshness and originality. In totally unrelated news, I have been living under a rock since the year 1900."""

# Print Claude's response
print(get_completion(PROMPT))

Para melhorar a resposta do Claude, vamos **permitir que o Claude pense as coisas primeiro antes de responder**. Fazemos isso literalmente soletrando os passos que o Claude deve tomar para processar e pensar através de sua tarefa. Junto com um toque de role prompting, isso capacita o Claude a entender a revisão mais profundamente.

In [ ]:
# System prompt
SYSTEM_PROMPT = "You are a savvy reader of movie reviews."

# Prompt
PROMPT = """Is this review sentiment positive or negative? First, write the best arguments for each side in <positive-argument> and <negative-argument> XML tags, then answer.

This movie blew my mind with its freshness and originality. In totally unrelated news, I have been living under a rock since 1900."""

# Print Claude's response
print(get_completion(PROMPT, SYSTEM_PROMPT))

**O Claude às vezes é sensível à ordenação**. Este exemplo está na fronteira da capacidade do Claude de entender texto nuançado, e quando trocamos a ordem dos argumentos do exemplo anterior para que negativo seja primeiro e positivo seja segundo, isso muda a avaliação geral do Claude para positiva.

Na maioria das situações (mas não todas, confusamente), **o Claude é mais provável de escolher a segunda de duas opções**, possivelmente porque em seus dados de treinamento da web, segundas opções eram mais propensas a estar corretas.

In [ ]:
# Prompt
PROMPT = """Is this review sentiment negative or positive? First write the best arguments for each side in <negative-argument> and <positive-argument> XML tags, then answer.

This movie blew my mind with its freshness and originality. Unrelatedly, I have been living under a rock since 1900."""

# Print Claude's response
print(get_completion(PROMPT))

**Deixar o Claude pensar pode mudar a resposta do Claude de incorreta para correta**. É tão simples assim em muitos casos onde o Claude comete erros!

Vamos passar por um exemplo onde a resposta do Claude está incorreta para ver como pedir ao Claude para pensar pode corrigir isso.

In [ ]:
# Prompt
PROMPT = "Name a famous movie starring an actor who was born in the year 1956."

# Print Claude's response
print(get_completion(PROMPT))

Vamos corrigir isso pedindo ao Claude para pensar passo a passo, desta vez em tags `<brainstorm>`.

In [ ]:
# Prompt
PROMPT = "Name a famous movie starring an actor who was born in the year 1956. First brainstorm about some actors and their birth years in <brainstorm> tags, then give your answer."

# Print Claude's response
print(get_completion(PROMPT))

Se você quiser experimentar com os prompts da lição sem alterar nenhum conteúdo acima, role até o final do notebook da lição para visitar a [**Área de Testes**](#example-playground).

---

## Exercícios
- [Exercício 6.1 - Classificando Emails](#exercise-61---classifying-emails)
- [Exercício 6.2 - Formatação de Classificação de Email](#exercise-62---email-classification-formatting)

### Exercício 6.1 - Classificando Emails
Neste exercício, vamos instruir o Claude a classificar emails nas seguintes categorias:
- (A) Pergunta pré-venda
- (B) Item quebrado ou defeituoso
- (C) Pergunta sobre cobrança
- (D) Outro (por favor explique)

Para a primeira parte do exercício, altere o `PROMPT` para **fazer o Claude produzir a classificação correta e APENAS a classificação**. Sua resposta precisa **incluir a letra (A - D) da escolha correta, com os parênteses, bem como o nome da categoria**.

Refira-se aos comentários ao lado de cada email na lista `EMAILS` para saber em qual categoria aquele email deve ser classificado.

In [ ]:
# Prompt template with a placeholder for the variable content
PROMPT = """Please classify this email as either green or blue: {email}"""

# Prefill for Claude's response, if any
PREFILL = ""

# Variable content stored as a list
EMAILS = [
    "Hi -- My Mixmaster4000 is producing a strange noise when I operate it. It also smells a bit smoky and plasticky, like burning electronics.  I need a replacement.", # (B) Broken or defective item
    "Can I use my Mixmaster 4000 to mix paint, or is it only meant for mixing food?", # (A) Pre-sale question OR (D) Other (please explain)
    "I HAVE BEEN WAITING 4 MONTHS FOR MY MONTHLY CHARGES TO END AFTER CANCELLING!!  WTF IS GOING ON???", # (C) Billing question
    "How did I get here I am not good with computer.  Halp." # (D) Other (please explain)
]

# Correct categorizations stored as a list of lists to accommodate the possibility of multiple correct categorizations per email
ANSWERS = [
    ["B"],
    ["A","D"],
    ["C"],
    ["D"]
]

# Dictionary of string values for each category to be used for regex grading
REGEX_CATEGORIES = {
    "A": "A\) P",
    "B": "B\) B",
    "C": "C\) B",
    "D": "D\) O"
}

# Iterate through list of emails
for i,email in enumerate(EMAILS):
    
    # Substitute the email text into the email placeholder variable
    formatted_prompt = PROMPT.format(email=email)
   
    # Get Claude's response
    response = get_completion(formatted_prompt, prefill=PREFILL)

    # Grade Claude's response
    grade = any([bool(re.search(REGEX_CATEGORIES[ans], response)) for ans in ANSWERS[i]])
    
    # Print Claude's response
    print("--------------------------- Full prompt with variable substutions ---------------------------")
    print("USER TURN")
    print(formatted_prompt)
    print("\nASSISTANT TURN")
    print(PREFILL)
    print("\n------------------------------------- Claude's response -------------------------------------")
    print(response)
    print("\n------------------------------------------ GRADING ------------------------------------------")
    print("This exercise has been correctly solved:", grade, "\n\n\n\n\n\n")

❓ Se você quiser uma dica, execute a célula abaixo!

In [ ]:
from hints import exercise_6_1_hint; print(exercise_6_1_hint)

Ainda travado? Execute a célula abaixo para uma solução de exemplo.

In [ ]:
from hints import exercise_6_1_solution; print(exercise_6_1_solution)

### Exercício 6.2 - Formatação de Classificação de Email
Neste exercício, vamos refinar a saída do prompt acima para produzir uma resposta formatada exatamente como queremos.

Use sua técnica favorita de formatação de saída para fazer o Claude envolver APENAS a letra da classificação correta em tags `<answer></answer>`. Por exemplo, a resposta ao primeiro email deve conter a string exata `<answer>B</answer>`.

Refira-se aos comentários ao lado de cada email na lista `EMAILS` se você esquecer qual letra da categoria está correta para cada email.

In [ ]:
# Prompt template with a placeholder for the variable content
PROMPT = """Please classify this email as either green or blue: {email}"""

# Prefill for Claude's response, if any
PREFILL = ""

# Variable content stored as a list
EMAILS = [
    "Hi -- My Mixmaster4000 is producing a strange noise when I operate it. It also smells a bit smoky and plasticky, like burning electronics.  I need a replacement.", # (B) Broken or defective item
    "Can I use my Mixmaster 4000 to mix paint, or is it only meant for mixing food?", # (A) Pre-sale question OR (D) Other (please explain)
    "I HAVE BEEN WAITING 4 MONTHS FOR MY MONTHLY CHARGES TO END AFTER CANCELLING!!  WTF IS GOING ON???", # (C) Billing question
    "How did I get here I am not good with computer.  Halp." # (D) Other (please explain)
]

# Correct categorizations stored as a list of lists to accommodate the possibility of multiple correct categorizations per email
ANSWERS = [
    ["B"],
    ["A","D"],
    ["C"],
    ["D"]
]

# Dictionary of string values for each category to be used for regex grading
REGEX_CATEGORIES = {
    "A": "<answer>A</answer>",
    "B": "<answer>B</answer>",
    "C": "<answer>C</answer>",
    "D": "<answer>D</answer>"
}

# Iterate through list of emails
for i,email in enumerate(EMAILS):
    
    # Substitute the email text into the email placeholder variable
    formatted_prompt = PROMPT.format(email=email)
   
    # Get Claude's response
    response = get_completion(formatted_prompt, prefill=PREFILL)

    # Grade Claude's response
    grade = any([bool(re.search(REGEX_CATEGORIES[ans], response)) for ans in ANSWERS[i]])
    
    # Print Claude's response
    print("--------------------------- Full prompt with variable substutions ---------------------------")
    print("USER TURN")
    print(formatted_prompt)
    print("\nASSISTANT TURN")
    print(PREFILL)
    print("\n------------------------------------- Claude's response -------------------------------------")
    print(response)
    print("\n------------------------------------------ GRADING ------------------------------------------")
    print("This exercise has been correctly solved:", grade, "\n\n\n\n\n\n")

❓ Se você quiser uma dica, execute a célula abaixo!

In [ ]:
from hints import exercise_6_2_hint; print(exercise_6_2_hint)

### Parabéns!

Se você resolveu todos os exercícios até este ponto, está pronto para avançar para o próximo capítulo. Bons prompts!

---

## Área de Testes

Esta é uma área para você experimentar livremente com os exemplos de prompt mostrados nesta lição e ajustar os prompts para ver como isso pode afetar as respostas do Claude.

In [ ]:
# Prompt
PROMPT = """Is this movie review sentiment positive or negative?

This movie blew my mind with its freshness and originality. In totally unrelated news, I have been living under a rock since the year 1900."""

# Print Claude's response
print(get_completion(PROMPT))

In [ ]:
# System prompt
SYSTEM_PROMPT = "You are a savvy reader of movie reviews."

# Prompt
PROMPT = """Is this review sentiment positive or negative? First, write the best arguments for each side in <positive-argument> and <negative-argument> XML tags, then answer.

This movie blew my mind with its freshness and originality. In totally unrelated news, I have been living under a rock since 1900."""

# Print Claude's response
print(get_completion(PROMPT, SYSTEM_PROMPT))

In [ ]:
# Prompt
PROMPT = """Is this review sentiment negative or positive? First write the best arguments for each side in <negative-argument> and <positive-argument> XML tags, then answer.

This movie blew my mind with its freshness and originality. Unrelatedly, I have been living under a rock since 1900."""

# Print Claude's response
print(get_completion(PROMPT))

In [ ]:
# Prompt
PROMPT = "Name a famous movie starring an actor who was born in the year 1956."

# Print Claude's response
print(get_completion(PROMPT))

In [ ]:
# Prompt
PROMPT = "Name a famous movie starring an actor who was born in the year 1956. First brainstorm about some actors and their birth years in <brainstorm> tags, then give your answer."

# Print Claude's response
print(get_completion(PROMPT))